In [ ]:
# 
# import dias.rewriter
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import os
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    import pandas as pd
import seaborn as sns  # data visualization
import matplotlib.pyplot as plt
import time

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/home/dias-benchmarks/notebooks/ampiiere/animal-crossing-villager-popularity-analysis/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
### cell 0 ###

vlgr_df = pd.read_csv("/home/dias-benchmarks/notebooks/ampiiere/animal-crossing-villager-popularity-analysis/input/animal-crossing-new-horizons-nookplaza-dataset/villagers.csv")
popul_df = pd.read_csv("/home/dias-benchmarks/notebooks/ampiiere/animal-crossing-villager-popularity-analysis/input/acnh-villager-popularity/acnh_villager_data.csv")

factor = 100

# Build a positional index that tiles the DataFrame factor times
n_pop = len(popul_df)
popul_idx = pd.Series(range(n_pop * factor)) % n_pop
popul_df = popul_df.take(popul_idx).reset_index(drop=True)

n_vlgr = len(vlgr_df)
vlgr_idx = pd.Series(range(n_vlgr * factor)) % n_vlgr
vlgr_df = vlgr_df.take(vlgr_idx).reset_index(drop=True)

# Info calls remain the same
print(popul_df.info())
vlgr_df.info()

In [ ]:
### cell 1 ###

vlgr_df.head()

In [ ]:
### cell 2 ###

popul_df.head()

In [ ]:
### cell 3 ###

vlgr_df.info()

In [ ]:
### cell 4 ###

popul_df.info()

In [ ]:
### cell 5 ###

# There are some missing/non-matching names 
vlgr_df["Name"].isin(popul_df['name']).sum()

In [ ]:
### cell 6 ###

# vlgr_df does not have these names...
mismatch_names = popul_df["name"][popul_df["name"].isin(vlgr_df["Name"]) == False]
mismatch_names

In [ ]:
### cell 7 ###

popul_df['name'] = popul_df['name'].replace({
    'OHare': "O'Hare",
    'Buck(Brows)': 'Buck',
    'Renee': 'Renée',
    'WartJr': 'Wart Jr.',
    'Crackle(Spork)': 'Spork'
})

In [ ]:
### cell 8 ###

# Checking if All names match
vlgr_df["Name"].isin(popul_df['name']).sum()

In [ ]:
### cell 9 ###

popul_df = popul_df[popul_df['name'].isin(vlgr_df['Name'])]

In [ ]:
### cell 10 ###

# Now that both df have same length, we can set index as names and combine the 2 dfs
popul_df.set_index('name', drop=True, inplace=True)
vlgr_df.set_index('Name', drop=True, inplace=True)

In [ ]:
### cell 11 ###

combined_df = popul_df.merge(vlgr_df, left_index=True, right_index=True)

In [ ]:
### cell 12 ###

# drop irrelevent columns
combined_df.drop(columns=['Furniture List', 'Filename', 'Unique Entry ID', "Wallpaper", "Flooring", "Birthday", "Favorite Song"], inplace=True)

In [ ]:
### cell 13 ###

# sort on GPU, return a new cudf‐backed DataFrame
combined_df = combined_df.sort_values(['tier', 'rank'])
# insert a 1..N ranking column at position 2 in one shot (GPU vectorized)
combined_df.insert(2,
                   'overall_ranking',
                   np.arange(1, len(combined_df) + 1))

In [ ]:
### cell 14 ###

overall_mean = combined_df.overall_ranking.mean()
print(f'The overall_mean is {overall_mean}, this would serve as a baseline for to compare against popularity performance of our features.')

In [ ]:
### cell 15 ###

combined_df.columns

In [ ]:
### cell 16 ###

combined_df['Gender'].value_counts()

In [ ]:
### cell 17 ###

# Compute the same output as groupby('tier').Gender.value_counts() using only GPU ops
tier_gender_counts = (
    combined_df
      .groupby(["tier", "Gender"])    # GPU
      .size()                             # GPU
      .reset_index(name="count")        # GPU
      .sort_values(["tier", "count"], ascending=[True, False])  # GPU
      .set_index(["tier", "Gender"])["count"]                 # GPU
      .rename("Gender")                 # GPU
)

tier_gender_counts

In [ ]:
### cell 18 ###

# Wrap the chain in parentheses so Python can parse the line breaks correctly
(
    combined_df
    .groupby(['tier', 'Gender'])['Catchphrase']
    .count()
    .unstack('Gender')
)

In [ ]:
### cell 19 ###

species_ranking = (
    combined_df[['Species', 'overall_ranking']]
    .groupby('Species')
    .mean()
    .reset_index()
    .sort_values('overall_ranking')
)
species_ranking

In [ ]:
### cell 20 ###

combined_df.Personality.value_counts()

In [ ]:
### cell 21 ###

personality_ranking = (
    combined_df
        .groupby('Personality')['overall_ranking']
        .mean()
        .reset_index()
        .sort_values('overall_ranking')
)

In [ ]:
### cell 22 ###

combined_df.groupby(['tier', 'Personality'])['Catchphrase'].count().unstack()

In [ ]:
### cell 23 ###

# generating value counts dataframe for each style type with fewer operations
style_ranking1 = (
    combined_df[['Style 1', 'overall_ranking']]
    .groupby('Style 1', as_index=False)
    .mean()
    .sort_values('overall_ranking')
)

style_ranking2 = (
    combined_df[['Style 2', 'overall_ranking']]
    .groupby('Style 2', as_index=False)
    .mean()
    .sort_values('overall_ranking')
)

In [ ]:
### cell 24 ###

# combining the 2 style columns and finding a mean in one vectorized step
style_ranking = style_ranking1
style_ranking['overall_ranking'] = (
    style_ranking['overall_ranking'] + style_ranking2['overall_ranking']
) / 2

In [ ]:
### cell 25 ###

style_ranking

In [ ]:
### cell 26 ###

combined_df.groupby(['tier', 'Style 1'])['Catchphrase'].count().unstack()

In [ ]:
### cell 27 ###

combined_df.groupby(['tier', 'Style 2'])['Catchphrase'].count().unstack()